In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
train=pd.read_csv('/Users/taha/Downloads/playground-series-s4e6/train.csv')
test=pd.read_csv('/Users/taha/Downloads/playground-series-s4e6/test.csv')

In [24]:
# Séparation de X et y sur l'intégralité du jeu d'entraînement
X = train.drop(columns=["id", "Target"])
y = train["Target"]

# Encodage de la variable cible sur TOUT le jeu 'y'
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_encoded = le.fit_transform(y)
# Ordre alphabétique : Dropout=0, Enrolled=1, Graduate=2

In [25]:
print("Proportions y_train :\n", y_train.value_counts(normalize=True))
print("10 premiers indices X_train :", X_train.index[:10])

Proportions y_train :
 Target
Graduate    0.474156
Dropout     0.330594
Enrolled    0.195249
Name: proportion, dtype: float64
10 premiers indices X_train : Index([75923, 50969, 57555, 62378, 41157, 19814, 47138, 62292, 59664, 75773], dtype='int64')


In [26]:
X.nunique()

Marital status                                       6
Application mode                                    22
Application order                                    8
Course                                              19
Daytime/evening attendance                           2
Previous qualification                              21
Previous qualification (grade)                     110
Nacionality                                         18
Mother's qualification                              35
Father's qualification                              39
Mother's occupation                                 40
Father's occupation                                 56
Admission grade                                    668
Displaced                                            2
Educational special needs                            2
Debtor                                               2
Tuition fees up to date                              2
Gender                                               2
Scholarshi

In [27]:
cat_cols = [
    'Marital status', 'Application mode', 'Course', 'Previous qualification', 
    'Nacionality', "Mother's qualification", "Father's qualification", 
    "Mother's occupation", "Father's occupation", 'Displaced', 
    'Educational special needs', 'Debtor', 'Tuition fees up to date', 
    'Gender', 'Scholarship holder', 'International'
]

In [28]:
def feature_engineering(df):
    df_feat = df.copy()
    
    # 1. Ratios de réussite académique
    df_feat['rate_1st_sem'] = df_feat['Curricular units 1st sem (approved)'] / (df_feat['Curricular units 1st sem (enrolled)'] + 1e-5)
    df_feat['rate_2nd_sem'] = df_feat['Curricular units 2nd sem (approved)'] / (df_feat['Curricular units 2nd sem (enrolled)'] + 1e-5)
    
    # 2. Cumuls annuels
    df_feat['total_approved'] = df_feat['Curricular units 1st sem (approved)'] + df_feat['Curricular units 2nd sem (approved)']
    df_feat['total_enrolled'] = df_feat['Curricular units 1st sem (enrolled)'] + df_feat['Curricular units 2nd sem (enrolled)']
    df_feat['overall_approval_rate'] = df_feat['total_approved'] / (df_feat['total_enrolled'] + 1e-5)
    
    # 3. Progression entre semestres
    df_feat['grade_progression'] = df_feat['Curricular units 2nd sem (grade)'] - df_feat['Curricular units 1st sem (grade)']
    df_feat['approved_progression'] = df_feat['Curricular units 2nd sem (approved)'] - df_feat['Curricular units 1st sem (approved)']
    
    # 4. Instabilité financière (Positif = à risque)
    df_feat['financial_instability'] = (df_feat['Debtor'] == 1).astype(int) - (df_feat['Tuition fees up to date'] == 1).astype(int) - (df_feat['Scholarship holder'] == 1).astype(int)
    
    return df_feat

# Application sur X et sur le jeu de test complet
X_engineered = feature_engineering(X)
X_test = test.drop(columns=["id"], errors="ignore")
X_test_engineered = feature_engineering(X_test)

# Conversion des variables catégorielles pour les modèles
for col in cat_cols:
    X_engineered[col] = X_engineered[col].astype('category')
    X_test_engineered[col] = X_test_engineered[col].astype('category')

In [29]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import lightgbm as lgb
from catboost import CatBoostClassifier

# Initialisation de la validation croisée
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Matrices pour stocker les probabilités sur l'entraînement (Out-Of-Fold) et le test
lgb_oof_probs = np.zeros((len(X_engineered), 3))
cb_oof_probs = np.zeros((len(X_engineered), 3))

lgb_test_probs = np.zeros((len(X_test_engineered), 3))
cb_test_probs = np.zeros((len(X_test_engineered), 3))

print("Début de l'entraînement 5-Fold sur 100% des données...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_engineered, y_encoded)):
    print(f"\n--- Pli {fold + 1}/5 ---")
    X_tr, y_tr = X_engineered.iloc[train_idx], y_encoded[train_idx]
    X_va, y_va = X_engineered.iloc[val_idx], y_encoded[val_idx]
    
    # Modèle LightGBM (avec paramètres optimisés)
    lgb_fold = lgb.LGBMClassifier(
        n_estimators=700, learning_rate=0.03, num_leaves=31, max_depth=8,
        subsample=0.8, colsample_bytree=0.8, objective='multiclass', num_class=3,
        random_state=42 + fold, verbosity=-1
    )
    lgb_fold.fit(X_tr, y_tr)
    lgb_oof_probs[val_idx] = lgb_fold.predict_proba(X_va)
    lgb_test_probs += lgb_fold.predict_proba(X_test_engineered) / skf.n_splits
    
    # Modèle CatBoost
    cb_fold = CatBoostClassifier(
        iterations=700, learning_rate=0.03, depth=6, loss_function='MultiClass',
        random_seed=42 + fold, verbose=0, cat_features=cat_cols
    )
    cb_fold.fit(X_tr, y_tr)
    cb_oof_probs[val_idx] = cb_fold.predict_proba(X_va)
    cb_test_probs += cb_fold.predict_proba(X_test_engineered) / skf.n_splits

print("\nEntraînement terminé ! Recherche du meilleur poids de Blend...")

# Optimisation du poids 'w' sur l'ensemble des prédictions (Out-Of-Fold)
best_w = 0.5
best_oof_acc = 0.0

for w in np.linspace(0, 1, 101):
    blended_oof = w * cb_oof_probs + (1 - w) * lgb_oof_probs
    acc = accuracy_score(y_encoded, np.argmax(blended_oof, axis=1))
    if acc > best_oof_acc:
        best_oof_acc = acc
        best_w = w

print(f"Meilleur Poids -> CatBoost: {best_w:.2f} | LightGBM: {1-best_w:.2f}")
print(f"Accuracy finale estimée (Blend OOF) : {best_oof_acc:.5f}")

# --- Génération du Fichier de Soumission Kaggle ---
# Application des meilleurs poids sur les probabilités du jeu de test
final_test_probs = best_w * cb_test_probs + (1 - best_w) * lgb_test_probs
final_preds_encoded = np.argmax(final_test_probs, axis=1)

# Reconversion des 0, 1, 2 en 'Dropout', 'Enrolled', 'Graduate'
final_preds_labels = le.inverse_transform(final_preds_encoded)

submission_final = pd.DataFrame({
    'id': test['id'],
    'Target': final_preds_labels
})

submission_final.to_csv('submission_final_0836.csv', index=False)
print("\nFichier submission_final_0836.csv généré avec succès ! Prêt pour Kaggle.")

Début de l'entraînement 5-Fold sur 100% des données...

--- Pli 1/5 ---

--- Pli 2/5 ---

--- Pli 3/5 ---

--- Pli 4/5 ---

--- Pli 5/5 ---

Entraînement terminé ! Recherche du meilleur poids de Blend...
Meilleur Poids -> CatBoost: 0.14 | LightGBM: 0.86
Accuracy finale estimée (Blend OOF) : 0.83252

Fichier submission_final_0836.csv généré avec succès ! Prêt pour Kaggle.
